In [1]:
import sys
import csv
import json
import hashlib
import subprocess
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)

for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1
    )

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow([
            "distance_km",
            "prep_time_min",
            "traffic_level",
            "rain",
            "delivery_min"
        ])

        for i in range(N_ROWS):
            w.writerow([
                distance_km[i],
                int(prep_time_min[i]),
                int(traffic_level[i]),
                int(rain[i]),
                delivery_min[i]
            ])

    return path

if not DATA.exists():
    make_delivery_csv()

print("dataset ready:", DATA)

print("Python version:", sys.version.split()[0])
print("Python program:", sys.executable)
print("Inside .venv :", ".venv" in sys.executable)

LIBRARIES = ["numpy", "pandas", "scikit-learn", "matplotlib"]

for name in LIBRARIES:
    print(name, version(name))

lines = [f"{name}=={version(name)}" for name in LIBRARIES]

Path("requirements.txt").write_text(
    "\n".join(lines) + "\n"
)

print(Path("requirements.txt").read_text())

careless = np.random.default_rng()
print(np.round(careless.uniform(0, 10, 3), 2))

first = np.random.default_rng(42).uniform(0, 10, 3)
second = np.random.default_rng(42).uniform(0, 10, 3)

print(np.round(first, 2))
print(np.round(second, 2))
print("identical:", np.array_equal(first, second))

def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

run_a = Path("run_a.csv")
run_b = Path("run_b.csv")

make_delivery_csv(run_a)
make_delivery_csv(run_b)

print(
    "identical files:",
    sha256_of(run_a) == sha256_of(run_b)
)

orders = pd.read_csv(DATA)

print(orders.shape)
print(orders.head())
print(orders.describe().round(1))

run_info = {
    "python": sys.version.split()[0],
    "seed": SEED,
    "rows": len(orders),
    "data_sha256": sha256_of(DATA),
    "libraries": {n: version(n) for n in LIBRARIES}
}

Path("run_info.json").write_text(
    json.dumps(run_info, indent=2)
)

print(json.dumps(run_info, indent=2))

run_a.unlink()
run_b.unlink()

distance_rng = np.random.default_rng(7)
distance_values = np.round(
    distance_rng.uniform(0.5, 12.0, N_ROWS), 2
)

print("T1 first three values:", distance_values[:3])

my_requirements = [
    f"numpy=={version('numpy')}",
    f"pandas=={version('pandas')}",
    f"scikit-learn=={version('scikit-learn')}"
]

Path("work").mkdir(exist_ok=True)

Path("work/my_requirements.txt").write_text(
    "\n".join(my_requirements) + "\n"
)

print("T2 requirements:")
print(Path("work/my_requirements.txt").read_text())

def fingerprint(path):
    data = pd.read_csv(path)
    return {
        "rows": len(data),
        "sha256": sha256_of(path),
        "seed": SEED
    }

print("T3 fingerprint:")
print(fingerprint(DATA))

def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=Path("."),
        capture_output=True,
        text=True
    )
    print(result.stdout + result.stderr)

git("init", "-q")
git("config", "user.name", "SCSE3040 Student")
git("config", "user.email", "student@bennett.edu.in")
git(
    "add",
    "MLOPs-Lab 1.ipynb",
    "requirements.txt",
    "run_info.json",
    "data/delivery_times.csv"
)
git("commit", "-q", "-m", "P01: pinned requirements and run record")
git("log", "--oneline")

print("\nFinal project structure:")

for p in Path(".").rglob("*"):
    if ".git" not in p.parts:
        print(p)

Python: 3.13.5
Folder: mlops1.ipynb
numpy - ok
pandas - ok
sklearn - ok
dataset ready: data\delivery_times.csv
Python version: 3.13.5
Python program: c:\ProgramData\anaconda3\python.exe
Inside .venv : False
numpy 2.4.2
pandas 2.3.1
scikit-learn 1.7.1
matplotlib 3.10.5
numpy==2.4.2
pandas==2.3.1
scikit-learn==1.7.1
matplotlib==3.10.5

[0.43 9.34 6.51]
[7.74 4.39 8.59]
[7.74 4.39 8.59]
identical: True
identical files: True
(600, 5)
   distance_km  prep_time_min  traffic_level  rain  delivery_min
0         9.40             17              1     0          51.3
1         5.55             24              2     1          54.2
2        10.37             28              3     0          67.7
3         8.52             23              2     0          51.2
4         1.58             29              2     1          42.1
       distance_km  prep_time_min  traffic_level   rain  delivery_min
count        600.0          600.0          600.0  600.0         600.0
mean           6.2           17.6   